# ZS-BERT / FewRel / m=5 / 5 independent runs

**Cách chạy:** Import notebook vào Kaggle, bật **Internet** và **GPU**, Add Input chứa
`fewrel_dataset.json` (hoặc `fewrel_all.json`, đủ 80 quan hệ × 700 mẫu), rồi **Run All**.
Mỗi notebook độc lập, tự cài thư viện và tải code/model; không cần upload file Python khác.
Huấn luyện dùng GPU đầu tiên, không nhân batch khi có T4 x2.

- **5 lần chạy:** seed `300, 301, 302, 303, 304`; mỗi seed khởi tạo model/Adam mới,
  chọn lại 5 quan hệ unseen; train có 75 quan hệ / 52,500 mẫu,
  test có 5 quan hệ / 3,500 mẫu. Seen và unseen luôn rời nhau.
- Adam, lr `5e-6`, batch `4`, gamma `7.5`, alpha `0.4`, inner product;
  `bert-large-cased` và SBERT `bert-large-nli-mean-tokens`, vector quan hệ 1024 chiều,
  SBERT cố định. FP32, không AMP, không tăng batch hay tích lũy gradient.
- 10 epoch theo lệnh chạy trong repository tác giả. Kết quả chính là
  **trung bình P/R/F1 của best epoch của từng run**, giống cách script tác giả theo dõi best.
  Đồng thời xuất **epoch cuối** để có kết quả không chọn epoch bằng unseen test.
  Độ lệch chuẩn dùng mẫu (`ddof=1`). Không lấy run tốt nhất trong 5 runs.
- Mốc FewRel trong Table 2 (P/R/F1, %): **76.96 / 78.86 / 77.90**.

**Phạm vi tái lập:** báo §5.1 công bố 5 random splits nhưng không nêu danh sách seed;
bộ seed ở đây là lựa chọn tái lập, không phải khẳng định seed gốc. Báo ghi hidden size 768,
còn code tác giả và notebook bạn gửi dùng BERT-large hidden 1024; notebook này theo code
để giữ kiến trúc đang dùng, không tự đổi sang BERT-base. 10 epoch lấy từ README/code,
không phải một giá trị được nêu trong §5.1. Không bảo đảm trùng điểm công bố.

Nguồn đối chiếu: [bài báo, §4–5 / Table 2](https://aclanthology.org/2021.naacl-main.272/),
[code tác giả](https://github.com/dinobby/ZS-BERT),
[repository của notebook đầu vào](https://github.com/Student-persist-1919/zs-bert-simulation).


## 1. Cấu hình và môi trường

Mặc định lưu checkpoint mỗi 1.000 batch và sau mỗi epoch (model + Adam + RNG + vị trí batch).
Một checkpoint đang chạy dùng khoảng vài GB; chỉ giữ một checkpoint, xóa khi run hoàn tất.
Nếu Kaggle ngắt phiên: lưu/tải outputs trước khi mất session, Add Input outputs cũ,
gán `RESUME_DIR` tới thư mục `results_m_*` bên dưới `/kaggle/input`, rồi Run All.
Việc tiếp tục chỉ xảy ra khi checksum dữ liệu, mã chạy và cấu hình trùng khớp.
Kaggle không tự bảo đảm giữ file chưa lưu khi phiên bị ngắt.

Dữ liệu dạng dict thực thể `{name,id,pos:[start,end]}`: mặc định `end` là exclusive,
đúng định dạng notebook đầu vào. Nếu dict.pos là danh sách chỉ số, đặt
`DICT_POSITION_FORMAT="indices"`. Định dạng FewRel gốc `[name,id,[[indices]]]` được giữ nguyên.


In [ ]:
import os
import sys
import subprocess
import json
import hashlib
import shutil
from pathlib import Path

M = 5
SEEDS = [300, 301, 302, 303, 304]
EPOCHS = 10
DATA_FILE = None  # Hoặc đường dẫn cụ thể dưới /kaggle/input.
DICT_POSITION_FORMAT = "span"  # "span" hoặc "indices" cho entity dạng dict.
RESUME_DIR = None  # Ví dụ: "/kaggle/input/my-old-output/results_m_5"
GRADIENT_CHECKPOINTING = True  # Giảm VRAM, giữ FP32 / batch=4 / loss gốc.
EVAL_BATCH_SIZE = 4  # Batch test nhỏ để an toàn VRAM, không ảnh hưởng batch train.
CHECKPOINT_EVERY = 1000
LOG_EVERY = 250

os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["HF_HUB_DISABLE_XET"] = "1"
WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
TASK_DIR = WORK_DIR / f"zs_bert_m_{M}_sources"
REPO_DIR = TASK_DIR / "repo"
RESULT_DIR = WORK_DIR / f"results_m_{M}"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
REPO_URL = "https://github.com/Student-persist-1919/zs-bert-simulation.git"
REPO_REV = "7a37cf51435b8ab8726d74ad1f9f374fbbae88b0"
AUTHOR_REV = "e2ffaea66244b2905055ee7cc47917985c3eab60"
PROPERTY_SHA256 = "d9c91ac01f18398e58a48ad7913e3f168553e3c06b306879b1b2389bde3eb21d"
BERT_ID = "google-bert/bert-large-cased"
BERT_REV = "06fa25dc1b6e034e91607fd67c247051ccaeff65"
SBERT_ID = "sentence-transformers/bert-large-nli-mean-tokens"
SBERT_REV = "a85b0d10e16867eb768e411c66f79475505b65dd"
PAPER_TARGET = dict(zip(["precision", "recall", "f1"], [76.96, 78.86, 77.9]))  # percent

# Cài trước khi import transformers / sentence_transformers để tránh module cũ trong kernel.
pinned = {"transformers": "4.44.2", "sentence-transformers": "3.0.1"}
import importlib.metadata
for package, version in pinned.items():
    module = package.replace("-", "_")
    if module in sys.modules and importlib.metadata.version(package) != version:
        raise RuntimeError(f"Restart Session trước khi Run All: {package} đã được import ở phiên bản khác.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                *[f"{p}=={v}" for p, v in pinned.items()], "lxml", "beautifulsoup4"], check=True)
import torch
print("Python:", sys.version.split()[0], "PyTorch:", torch.__version__, flush=True)
from packaging.version import Version
if Version(torch.__version__.split("+")[0]) < Version("2.4"):
    raise RuntimeError("Notebook cần PyTorch >=2.4; hãy dùng Kaggle GPU image mới.")
if not torch.cuda.is_available():
    raise RuntimeError("Hãy bật GPU trong Kaggle Settings rồi Restart Session và Run All.")
print("GPU:", torch.cuda.get_device_name(0), flush=True)
print("Kết quả:", RESULT_DIR, flush=True)
assert len(SEEDS) == 5 and len(set(SEEDS)) == 5
assert M in (5, 10, 15)


## 2. Mã nguồn và model được cố định phiên bản

Giữ model/loss và cách đánh dấu entity của implementation đầu vào (cùng cách làm của code tác giả).
Implementation này đánh dấu entity theo chỉ số token gốc +1, không căn chỉnh lại khi WordPiece tách từ;
đây là giới hạn được giữ để tránh âm thầm đổi mô hình đang tái lập.
Dùng thứ tự nhãn xác định thay vì `set`, dùng exhaustive argmax inner product thay BallTree
(negative inner product không thỏa điều kiện metric để BallTree cắt tỉa an toàn).
Cache tokenize/SBERT chỉ tiết kiệm công việc lặp, không học từ test.


In [ ]:
TASK_DIR.mkdir(parents=True, exist_ok=True)
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(REPO_DIR)], check=True)
actual = subprocess.check_output(["git", "-C", str(REPO_DIR), "remote", "get-url", "origin"], text=True).strip()
if actual != REPO_URL:
    raise RuntimeError(f"Sai origin repository: {actual}")
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", REPO_REV], check=True)
assert subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip() == REPO_REV

import urllib.request
PROPERTY_PATH = TASK_DIR / "author_property_list.html"
property_url = f"https://raw.githubusercontent.com/dinobby/ZS-BERT/{AUTHOR_REV}/resources/property_list.html"
if not PROPERTY_PATH.exists():
    PROPERTY_PATH.write_bytes(urllib.request.urlopen(property_url, timeout=120).read())
if hashlib.sha256(PROPERTY_PATH.read_bytes()).hexdigest() != PROPERTY_SHA256:
    raise RuntimeError("Checksum bảng mô tả gốc không khớp snapshot tác giả.")

from huggingface_hub import snapshot_download
download_args = dict(allow_patterns=["*.json", "vocab.txt", "model.safetensors"],
                     ignore_patterns=["onnx/*", "openvino/*"])
BERT_PATH = snapshot_download(BERT_ID, revision=BERT_REV, **download_args)
SBERT_PATH = snapshot_download(SBERT_ID, revision=SBERT_REV, **download_args)

# Cache tokenizer qua đường dẫn model đã khóa revision, không gọi lại nhánh "main".
helper_path = REPO_DIR / "model/data_helper.py"
helper = helper_path.read_text(encoding="utf-8")
old = '"bert-large-cased", do_lower_case=False'
new = 'os.environ["ZS_BERT_TOKENIZER_PATH"], do_lower_case=False'
if old not in helper and new not in helper:
    raise RuntimeError("Không tìm thấy đoạn tokenizer mong đợi; dừng để tránh patch sai.")
helper_path.write_text(helper.replace(old, new), encoding="utf-8")
os.environ["ZS_BERT_TOKENIZER_PATH"] = BERT_PATH
print("Repository + model revisions locked.", flush=True)


## 3. Kiểm tra và chuẩn hóa FewRel

Dừng nếu thiếu dữ liệu, trùng nhiều file đầu vào, entity sai vị trí hoặc không đủ
56.000 mẫu / 80 quan hệ. Không tự thay thế bằng dữ liệu thử hoặc bỏ mẫu lỗi.


In [ ]:
"""Normalize the user's FewRel format without guessing native index lists as spans."""
from collections import defaultdict
import json
from pathlib import Path


def normalize_entity(entity, tokens, dict_position_format='span'):
    if isinstance(entity, dict):
        name, qid, positions = entity.get('name', ''), entity.get('id', 'Q0'), entity['pos']
        if positions and isinstance(positions[0], list):
            indices = positions[0]  # explicitly nested index list
        elif dict_position_format == 'span':
            if len(positions) != 2:
                raise ValueError('Expected dict.pos=[start,end_exclusive]; set DICT_POSITION_FORMAT="indices" if needed.')
            indices = list(range(*positions))
        else:
            indices = positions
    else:
        name, qid, positions = entity
        indices = positions[0]  # official FewRel [name,id,[[token indices], ...]]
    if not indices or any(type(i) is not int or not 0 <= i < len(tokens) for i in indices):
        raise ValueError(f'Invalid entity positions: {indices!r}; token count={len(tokens)}')
    if indices != sorted(set(indices)):
        raise ValueError(f'Entity positions must be unique and sorted: {indices!r}')
    return [name, qid, [indices]]


def preprocess(source, destination, dict_position_format='span'):
    raw = json.loads(Path(source).read_text(encoding='utf-8'))
    if isinstance(raw, dict):
        rows = ((pid, item) for pid, items in raw.items() for item in items)
    elif isinstance(raw, list):
        rows = ((item['relation'], item) for item in raw)
    else:
        raise ValueError('Expected relation->instances dictionary or flat instance list.')
    grouped = defaultdict(list)
    for pid, item in rows:
        tokens = item.get('tokens', item.get('token'))
        if not tokens or not all(isinstance(token, str) for token in tokens):
            raise ValueError('Each sample must contain a nonempty list of tokens.')
        grouped[pid].append({'tokens': tokens, 'relation': pid,
                             'h': normalize_entity(item['h'], tokens, dict_position_format),
                             't': normalize_entity(item['t'], tokens, dict_position_format)})
    counts = {pid: len(items) for pid, items in grouped.items()}
    if len(counts) != 80 or set(counts.values()) != {700}:
        raise ValueError(f'Paper FewRel requires 80 relations x 700 samples = 56,000; got {len(counts)} relations, counts={counts}')
    Path(destination).write_text(json.dumps(grouped, ensure_ascii=False), encoding='utf-8')
    return counts


if DICT_POSITION_FORMAT not in ("span", "indices"):
    raise ValueError("DICT_POSITION_FORMAT phải là span hoặc indices")
if DATA_FILE is None:
    input_root = Path("/kaggle/input")
    candidates = sorted({p for name in ["fewrel_dataset.json", "fewrel_all.json"]
                         for p in input_root.rglob(name)}) if input_root.exists() else []
    if len(candidates) != 1:
        raise FileNotFoundError(f"Tìm thấy {len(candidates)} file FewRel: {candidates}. Gán DATA_FILE cụ thể ở cell cấu hình.")
    DATA_FILE = candidates[0]
DATA_FILE = Path(DATA_FILE)
DATA_PATH = TASK_DIR / "fewrel_all.json"
counts = preprocess(DATA_FILE, DATA_PATH, DICT_POSITION_FORMAT)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

print("Input:", DATA_FILE)
print("Checked:", len(counts), "relations;", sum(counts.values()), "samples")
print("Train/test:", (80-M)*700, "/", M*700, "samples per run")


## 4. Vector thuộc tính cố định và script chạy độc lập

SBERT encode mô tả từ snapshot `property_list.html` của **repository tác giả** (khóa commit + SHA256), ở chế độ eval;
không chuẩn hóa độ dài vector vì huấn luyện/suy luận dùng inner product gốc.
SBERT chạy trong process riêng để trả VRAM trước khi bắt đầu BERT.


In [ ]:
ATTR_SCRIPT = 'import json\nimport os\nfrom pathlib import Path\nimport sys\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom sentence_transformers import SentenceTransformer\ntorch.backends.cuda.matmul.allow_tf32 = False\ntorch.backends.cudnn.allow_tf32 = False\ntorch.manual_seed(0)\nlabels = list(json.loads(Path(sys.argv[1]).read_text(encoding="utf-8")))\ntable = pd.read_html(sys.argv[2])[0]\ndescriptions = {}\nfor pid in labels:\n    rows = table.loc[table.ID == pid, "description"]\n    if len(rows) != 1 or not isinstance(rows.iloc[0], str) or not rows.iloc[0].strip():\n        raise ValueError(f"Missing/ambiguous description: {pid}")\n    descriptions[pid] = rows.iloc[0]\nencoder = SentenceTransformer(sys.argv[3], device="cuda", model_kwargs={"attn_implementation": "eager"})\nencoder.eval()\nemb = encoder.encode([descriptions[k] for k in labels], batch_size=4,\n                     convert_to_numpy=True, normalize_embeddings=False, show_progress_bar=True)\nassert emb.shape == (80, 1024) and np.isfinite(emb).all()\nnp.savez(sys.argv[4], **{pid: row.astype("float32") for pid, row in zip(labels, emb)})\nPath(sys.argv[5]).write_text(json.dumps(descriptions, ensure_ascii=False, indent=2), encoding="utf-8")\n'
TRAINER_SOURCE = '"""Run one independently initialized FewRel experiment; invoked by each notebook."""\nimport argparse\nimport hashlib\nimport importlib.metadata\nimport json\nimport os\nfrom pathlib import Path\nimport random\nimport sys\nimport time\n\nimport numpy as np\nimport torch\nfrom torch.utils.data import DataLoader\nfrom transformers import BertConfig\n\nfrom data_helper import FewRelDataset, create_mini_batch\nfrom evaluation import compute_macro_PRF\nfrom model import ZSBert\n\n\ndef write_json(path, value):\n    path = Path(path)\n    tmp = path.with_suffix(path.suffix + \'.tmp\')\n    tmp.write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding=\'utf-8\')\n    tmp.replace(path)\n\n\ndef metrics(model, loader, attributes, gold, offset, device):\n    model.eval()\n    predictions = []\n    with torch.no_grad():\n        attr = torch.as_tensor(attributes, device=device)\n        for data in loader:\n            ids, segments, e1, e2, mask, rel, _ = data\n            _, emb = model(input_ids=ids.to(device), token_type_ids=segments.to(device),\n                           e1_mask=e1.to(device), e2_mask=e2.to(device),\n                           attention_mask=mask.to(device))\n            # Exact exhaustive nearest neighbour under negative inner product.\n            # Negative inner product is not a metric; BallTree pruning is invalid.\n            predictions.extend((emb @ attr.T).argmax(1).cpu().numpy() + offset)\n    predictions = np.asarray(predictions)\n    p, r, f1 = compute_macro_PRF(predictions, gold)\n    from sklearn.metrics import precision_recall_fscore_support\n    sp, sr, sf, _ = precision_recall_fscore_support(\n        gold, predictions, labels=np.arange(offset, offset + len(attributes)),\n        average=\'macro\', zero_division=0)\n    return {\'precision\': float(p), \'recall\': float(r), \'f1\': float(f1),\n            \'standard_macro_precision\': float(sp), \'standard_macro_recall\': float(sr),\n            \'standard_macro_f1\': float(sf)}\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--config\', required=True)\n    parser.add_argument(\'--seed\', required=True, type=int)\n    args = parser.parse_args()\n    cfg = json.loads(Path(args.config).read_text(encoding=\'utf-8\'))\n    seed = args.seed\n    out = Path(cfg[\'result_dir\'])\n    out.mkdir(parents=True, exist_ok=True)\n    result_file = out / f\'run_seed_{seed}.json\'\n    checkpoint_file = out / \'active_checkpoint.pt\'\n    if result_file.exists():\n        result = json.loads(result_file.read_text(encoding=\'utf-8\'))\n        if result.get(\'complete\') and result[\'config_hash\'] == cfg[\'config_hash\']:\n            print(f\'Seed {seed}: completed earlier, skipping.\', flush=True)\n            return\n        raise RuntimeError(f\'Existing result has a different config: {result_file}\')\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n    torch.backends.cuda.matmul.allow_tf32 = False\n    torch.backends.cudnn.allow_tf32 = False\n    torch.use_deterministic_algorithms(True)\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'Select a Kaggle GPU accelerator before running training.\')\n    device = torch.device(\'cuda:0\')\n    checkpoint = None\n    if checkpoint_file.exists():\n        # Only load locally produced/trusted notebook outputs, never arbitrary inputs.\n        checkpoint = torch.load(checkpoint_file, map_location=\'cpu\', weights_only=False)\n        if checkpoint[\'config_hash\'] != cfg[\'config_hash\'] or checkpoint[\'seed\'] != seed:\n            raise RuntimeError(\'Active checkpoint does not match config/seed. Restore all run JSONs too.\')\n\n    raw = json.loads(Path(cfg[\'data_path\']).read_text(encoding=\'utf-8\'))\n    all_keys = list(raw)  # Preserve the input\'s relation order, as in the original script.\n    test_keys = random.sample(all_keys, cfg[\'m\'])\n    train_keys = [key for key in all_keys if key not in test_keys]\n    assert not set(train_keys).intersection(test_keys)\n    # Deterministic label order; no Python set iteration dependence.\n    labels = train_keys + test_keys\n    property2idx = {pid: i for i, pid in enumerate(labels)}\n    attrs = np.load(cfg[\'attribute_path\'])\n    pid2vec = {pid: attrs[pid] for pid in labels}\n    train_data = [item for key in train_keys for item in raw[key]]\n    test_data = [item for key in test_keys for item in raw[key]]\n    print(f\'m={cfg["m"]}, seed={seed}, train={len(train_data)}, test={len(test_data)}\', flush=True)\n    print(\'Unseen relations:\', test_keys, flush=True)\n    trainset = FewRelDataset(\'train\', train_data, pid2vec, property2idx)\n    testset = FewRelDataset(\'test\', test_data, pid2vec, property2idx)\n    testloader = DataLoader(testset, batch_size=cfg[\'eval_batch_size\'],\n                            collate_fn=create_mini_batch, num_workers=0,\n                            generator=torch.Generator().manual_seed(seed))\n    test_attr = np.stack([pid2vec[key] for key in test_keys])\n    gold = np.array([property2idx[item[\'relation\']] for item in test_data])\n    config = BertConfig.from_pretrained(cfg[\'bert_path\'], num_labels=len(train_keys))\n    config.relation_emb_dim = 1024\n    config.margin, config.alpha, config.dist_func = cfg[\'gamma\'], cfg[\'alpha\'], \'inner\'\n    # Original implementation uses explicit attention, before SDPA became the default.\n    model = ZSBert.from_pretrained(cfg[\'bert_path\'], config=config,\n                                  attn_implementation=\'eager\').to(device)\n    if cfg[\'gradient_checkpointing\']:\n        model.bert.gradient_checkpointing_enable(gradient_checkpointing_kwargs={\'use_reentrant\': False})\n    optimizer = torch.optim.Adam(model.parameters(), lr=cfg[\'lr\'])\n    start_epoch, next_step, totals, history, prior_seconds = 0, 0, [0., 0], [], 0.\n    if checkpoint:\n        model.load_state_dict(checkpoint[\'model\'])\n        optimizer.load_state_dict(checkpoint[\'optimizer\'])\n        start_epoch, next_step = checkpoint[\'epoch\'], checkpoint[\'next_step\']\n        totals, history = checkpoint[\'totals\'], checkpoint[\'history\']\n        prior_seconds = checkpoint[\'elapsed_sec\']\n        random.setstate(checkpoint[\'python_rng\'])\n        np.random.set_state(checkpoint[\'numpy_rng\'])\n        torch.set_rng_state(checkpoint[\'torch_rng\'])\n        torch.cuda.set_rng_state_all(checkpoint[\'cuda_rng\'])\n        print(f\'Resuming epoch {start_epoch + 1}, next batch {next_step + 1}\', flush=True)\n        del checkpoint\n    started = time.time()\n\n    def save_checkpoint(epoch, step, running):\n        tmp = checkpoint_file.with_suffix(\'.tmp\')\n        state = {\'config_hash\': cfg[\'config_hash\'], \'seed\': seed,\n                 \'epoch\': epoch, \'next_step\': step, \'totals\': running, \'history\': history,\n                 \'elapsed_sec\': prior_seconds + time.time() - started,\n                 \'model\': model.state_dict(), \'optimizer\': optimizer.state_dict(),\n                 \'python_rng\': random.getstate(), \'numpy_rng\': np.random.get_state(),\n                 \'torch_rng\': torch.get_rng_state(), \'cuda_rng\': torch.cuda.get_rng_state_all()}\n        torch.save(state, tmp)\n        tmp.replace(checkpoint_file)\n        print(f\'Checkpoint saved (epoch={epoch + 1}, next_step={step}).\', flush=True)\n\n    for epoch in range(start_epoch, cfg[\'epochs\']):\n        model.train()\n        if epoch != start_epoch:\n            next_step, totals = 0, [0., 0]\n        # Fixed permutation per epoch allows skipping batches on resume without replaying dropout.\n        gen = torch.Generator().manual_seed(seed * 10000 + epoch)\n        order = torch.randperm(len(trainset), generator=gen).tolist()\n        loader = DataLoader(trainset, batch_size=cfg[\'batch_size\'],\n                            sampler=order[next_step * cfg[\'batch_size\']:],\n                            collate_fn=create_mini_batch, num_workers=0,\n                            generator=torch.Generator().manual_seed(seed * 10000 + epoch))\n        total_steps = (len(trainset) + cfg[\'batch_size\'] - 1) // cfg[\'batch_size\']\n        for step, data in enumerate(loader, start=next_step):\n            ids, segments, e1, e2, mask, rel, y = [t.to(device) for t in data]\n            optimizer.zero_grad(set_to_none=True)\n            outputs, _ = model(input_ids=ids, token_type_ids=segments, e1_mask=e1,\n                               e2_mask=e2, attention_mask=mask,\n                               input_relation_emb=rel, labels=y)\n            loss = outputs[0]\n            if not torch.isfinite(loss):\n                raise FloatingPointError(f\'Non-finite loss at epoch {epoch+1}, batch {step+1}\')\n            loss.backward()\n            optimizer.step()\n            totals[0] += loss.item()\n            totals[1] += 1\n            if (step + 1) % cfg[\'log_every\'] == 0 or step + 1 == total_steps:\n                print(f\'seed={seed} epoch={epoch+1}/{cfg["epochs"]} \'\n                      f\'batch={step+1}/{total_steps} loss={totals[0]/totals[1]:.6f}\', flush=True)\n            if (step + 1) % cfg[\'checkpoint_every\'] == 0 and step + 1 < total_steps:\n                save_checkpoint(epoch, step + 1, totals)\n        scores = metrics(model, testloader, test_attr, gold, len(train_keys), device)\n        row = {\'epoch\': epoch + 1, \'train_loss\': totals[0] / max(totals[1], 1), **scores}\n        history.append(row)\n        print(\'EVALUATION:\', json.dumps(row), flush=True)\n        write_json(out / f\'history_seed_{seed}.json\', history)\n        save_checkpoint(epoch + 1, 0, [0., 0])\n    best = max(history, key=lambda row: row[\'f1\'])\n    result = {\'complete\': True, \'config_hash\': cfg[\'config_hash\'], \'seed\': seed, \'m\': cfg[\'m\'],\n              \'train_relations\': train_keys, \'test_relations\': test_keys,\n              \'train_samples\': len(train_data), \'test_samples\': len(test_data),\n              \'best\': best, \'final\': history[-1], \'history\': history,\n              \'total_time_sec\': prior_seconds + time.time() - started,\n              \'metric_note\': \'Main P/R/F1 use authors compute_macro_PRF; standard macro metrics also saved.\',\n              \'selection_note\': \'best uses unseen TEST F1, matching authors script; final does not select on test.\',\n              \'runtime\': {p: importlib.metadata.version(p) for p in\n                          [\'torch\', \'transformers\', \'sentence-transformers\', \'numpy\', \'scikit-learn\']},\n              \'gpu\': torch.cuda.get_device_name(0)}\n    write_json(result_file, result)\n    checkpoint_file.unlink(missing_ok=True)\n    print(f\'COMPLETE seed={seed}: best={best["f1"]:.6f}, final={history[-1]["f1"]:.6f}\', flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n'
ATTR_PATH = TASK_DIR / "attributes.npz"
DESCRIPTION_PATH = RESULT_DIR / "relation_descriptions.json"
(TASK_DIR / "build_attributes.py").write_text(ATTR_SCRIPT, encoding="utf-8")
TRAINER_PATH = REPO_DIR / "model/train_five_runs.py"
TRAINER_PATH.write_text(TRAINER_SOURCE, encoding="utf-8")

subprocess.run([sys.executable, "-u", str(TASK_DIR / "build_attributes.py"),
                str(DATA_PATH), str(PROPERTY_PATH),
                SBERT_PATH, str(ATTR_PATH), str(DESCRIPTION_PATH)], check=True)

import importlib.metadata
signature = {
    "m": M, "seeds": SEEDS, "epochs": EPOCHS, "batch_size": 4,
    "lr": 5e-6, "gamma": 7.5, "alpha": 0.4, "precision": "fp32", "dist_func": "inner",
    "attention_implementation": "eager",
    "eval_batch_size": EVAL_BATCH_SIZE, "gradient_checkpointing": GRADIENT_CHECKPOINTING,
    "dict_position_format": DICT_POSITION_FORMAT, "repo_rev": REPO_REV,
    "author_revision": AUTHOR_REV, "property_sha256": PROPERTY_SHA256,
    "bert_revision": BERT_REV, "sbert_revision": SBERT_REV,
    "data_sha256": sha256_file(DATA_PATH),
    "descriptions_sha256": sha256_file(DESCRIPTION_PATH),
    "trainer_sha256": sha256_file(TRAINER_PATH),
    "helper_sha256": sha256_file(REPO_DIR / "model/data_helper.py"),
    "model_sha256": sha256_file(REPO_DIR / "model/model.py"),
    "gpu_model": torch.cuda.get_device_name(0),
    "packages": {p: importlib.metadata.version(p) for p in
                 ["torch", "transformers", "sentence-transformers", "numpy", "scikit-learn"]},
}
CONFIG_HASH = hashlib.sha256(json.dumps(signature, sort_keys=True).encode()).hexdigest()
CFG = {**signature, "config_hash": CONFIG_HASH, "data_path": str(DATA_PATH),
       "attribute_path": str(ATTR_PATH), "bert_path": BERT_PATH,
       "result_dir": str(RESULT_DIR), "checkpoint_every": CHECKPOINT_EVERY, "log_every": LOG_EVERY}
CONFIG_PATH = TASK_DIR / "run_config.json"
CONFIG_PATH.write_text(json.dumps(CFG, indent=2), encoding="utf-8")
(RESULT_DIR / "experiment_config.json").write_text(json.dumps(CFG, indent=2), encoding="utf-8")
print("Config hash:", CONFIG_HASH)


## 5. Chạy đủ 5 runs và lưu log trực tiếp

Các run chạy tuần tự trong process riêng; bộ nhớ/model/optimizer của run trước được giải phóng.
Checkpoint chứa cả Adam và trạng thái RNG; tiếp tục tại batch đã lưu.
Cần giữ cùng cấu hình, dữ liệu, runtime/GPU để tái lập sát nhất khi resume.
Log và kết quả từng run nằm trong `/kaggle/working/results_m_*`.


In [ ]:
if RESUME_DIR:
    resume = Path(RESUME_DIR)
    old_config = json.loads((resume / "experiment_config.json").read_text(encoding="utf-8"))
    if old_config["config_hash"] != CONFIG_HASH:
        raise RuntimeError("Outputs cũ không khớp dữ liệu/cấu hình/runtime hiện tại.")
    # Only restore trusted outputs created by this notebook.
    for pattern in ["run_seed_*.json", "history_seed_*.json", "active_checkpoint.pt", "train_seed_*.log"]:
        for src in resume.glob(pattern):
            dst = RESULT_DIR / src.name
            if src.resolve() != dst.resolve() and not dst.exists():
                shutil.copy2(src, dst)
    # Reuse exactly the saved fixed attributes, not a recomputation on another GPU.
    if (resume / "attributes.npz").exists():
        shutil.copy2(resume / "attributes.npz", ATTR_PATH)
# Bundle fixed attributes with outputs for continuation in another session.
shutil.copy2(ATTR_PATH, RESULT_DIR / "attributes.npz")
env = os.environ.copy()
env["PYTHONHASHSEED"] = "0"
for run_index, seed in enumerate(SEEDS, 1):
    print(f"\n{'='*60}\nRUN {run_index}/5 | m={M} | seed={seed}\n{'='*60}", flush=True)
    cmd = [sys.executable, "-u", str(TRAINER_PATH), "--config", str(CONFIG_PATH), "--seed", str(seed)]
    log_path = RESULT_DIR / f"train_seed_{seed}.log"
    with open(log_path, "a", encoding="utf-8") as log:
        process = subprocess.Popen(cmd, cwd=str(REPO_DIR / "model"), env=env,
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, encoding="utf-8", errors="replace", bufsize=1)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            code = process.wait()
        except BaseException:
            process.terminate()
            process.wait()
            raise
    if code:
        raise RuntimeError(f"Run seed={seed} thất bại (exit={code}). Xem {log_path}; checkpoint gần nhất còn được giữ.")
print("Hoàn tất 5 runs.", flush=True)


## 6. Tổng hợp 5 runs: mean ± sample std và chênh lệch so với báo

`best`: cùng cách chọn epoch của script tác giả, có dùng test F1 nên có thể lạc quan.
`final`: epoch 10, không dùng test để chọn checkpoint.
P/R/F1 chính dùng **hàm `compute_macro_PRF` của tác giả**: P chia cho số nhãn được dự đoán,
R chia cho số nhãn thật, F1 là harmonic mean của P và R; đây không phải sklearn macro-F1.
Các cột `standard_macro_*` bổ sung dùng sklearn trên toàn bộ m nhãn.
F1 trung bình là trung bình F1 từng run, không tính lại F1 từ P/R trung bình.
Mọi giá trị trong CSV tổng hợp là **phần trăm**, chênh lệch là **điểm phần trăm**.


In [ ]:
import csv
import statistics

results = []
for seed in SEEDS:
    path = RESULT_DIR / f"run_seed_{seed}.json"
    result = json.loads(path.read_text(encoding="utf-8"))
    assert result["complete"] and result["config_hash"] == CONFIG_HASH
    assert result["seed"] == seed and result["m"] == M
    assert len(result["history"]) == EPOCHS
    assert len(result["test_relations"]) == M
    assert not set(result["train_relations"]) & set(result["test_relations"])
    results.append(result)
assert len(results) == 5
splits = [tuple(sorted(r["test_relations"])) for r in results]
if len(set(splits)) < 5:
    print("Lưu ý: có split được chọn trùng ngẫu nhiên giữa các seed.")

metric_names = ["precision", "recall", "f1", "standard_macro_precision",
                "standard_macro_recall", "standard_macro_f1"]
per_run = []
aggregate = []
for mode in ["best", "final"]:
    print(f"\n{mode.upper()} -- P / R / F1 (%)")
    for result in results:
        row = result[mode]
        per_run.append({"m": M, "seed": result["seed"], "mode": mode,
                        "epoch": row["epoch"], **{key: 100*row[key] for key in metric_names}})
        print(f"seed {result['seed']}, epoch {row['epoch']}: "
              f"{100*row['precision']:.2f} / {100*row['recall']:.2f} / {100*row['f1']:.2f}")
    for key in metric_names:
        values = [100 * result[mode][key] for result in results]
        mean, std = statistics.mean(values), statistics.stdev(values)
        paper = PAPER_TARGET.get(key)
        aggregate.append({"m": M, "mode": mode, "metric": key, "runs": 5,
                          "mean_pct": mean, "sample_std_pct": std,
                          "paper_pct": paper, "delta_pp": mean-paper if paper is not None else None})
        print(f"{key}: {mean:.2f} ± {std:.2f}" +
              (f" | paper {paper:.2f} | delta {mean-paper:+.2f} pp" if paper is not None else ""))
for filename, rows in [("per_run_metrics.csv", per_run), ("aggregate_metrics.csv", aggregate)]:
    with open(RESULT_DIR / filename, "w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
(RESULT_DIR / "five_run_summary.json").write_text(json.dumps({
    "m": M, "seeds": SEEDS, "n_runs": 5, "std_ddof": 1, "unit": "percent",
    "paper_target_pct": PAPER_TARGET, "aggregate": aggregate,
    "per_run": per_run, "config_hash": CONFIG_HASH,
    "notes": ["best selects epoch by test F1, as in authors script",
              "final is fixed final epoch", "seeds 300-304 are reproduction choices, unpublished in paper",
              "paper hidden size 768 differs from authors BERT-large code (1024)"]
}, indent=2, ensure_ascii=False), encoding="utf-8")
print("\nĐã lưu JSON/CSV/logs tại:", RESULT_DIR)
for path in sorted(RESULT_DIR.iterdir()):
    print(path.name)
